# Урок 13 - Пам’ять агента


## Setup

This notebook demonstrates how to build a travel booking agent with **persistent memory** using the **Microsoft Agent Framework** (MAF).

You will learn how different types of agent memory — working, short-term, and long-term — affect how an agent retains and uses information across conversations.

**Prerequisites:**
- A Microsoft Foundry project with a deployed chat model (e.g. `gpt-4.1-mini`).
- Logged in with the Azure CLI — run `az login` in your terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Типи пам’яті агента

Штучні агенти можуть використовувати різні типи пам’яті, кожен із яких виконує свою окрему функцію:

### Робоча пам’ять
Сам ланцюжок розмови — повідомлення, обмінені в одній сесії. Агент може звертатися до попередніх повідомлень у тій самій розмові для підтримки послідовності. У MAF це створюється за допомогою **`agent.create_session()`**, що повертає `AgentSession`.

### Короткотривала пам’ять
Інформація, що зберігається протягом виконання завдання або сесії, але не зберігається постійно. Наприклад, агент може накопичувати факти під час багатокрокової планувальної розмови і використовувати їх для створення кінцевого маршруту.

### Довготривала пам’ять
Уподобання і факти, що зберігаються **між сесіями**. Повторному користувачеві не потрібно знову повідомляти свої дієтичні обмеження чи стиль подорожей. Довготривала пам’ять зазвичай підтримується зовнішнім сховищем — базою даних, файлом або векторним індексом — і надається агенту через інструменти.


## Робоча пам’ять із сесіями

Найпростіша форма пам’яті — сесія розмови. Коли ви передаєте ту ж сесію (створену за допомогою `agent.create_session()`) до послідовних викликів `agent.run()`, агент бачить повну історію цієї розмови і може пригадати раніші деталі.

Давайте створимо туристичного агента і продемонструємо робочу пам’ять.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Аґент правильно згадав бюджет, бо обидва повідомлення належать до однієї сесії. Це — **оперативна пам’ять** — існує лише протягом сесії.

### Що відбувається з новою ниткою?

Якщо ми створюємо **нову** сесію, аґент не має пам’яті про попередню розмову:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Патерн довготривалої пам’яті

Щоб запам’ятовувати вподобання користувача **поміж сесіями**, нам потрібне постійне сховище, яке існує поза межами потоку розмови. Аґент отримує доступ до цього сховища через **інструменти** — функції, які він може викликати, щоб зберігати та витягувати інформацію.

Нижче ми реалізуємо просте сховище вподобань у пам’яті (у продуктивному середовищі ви б підтримували це у базі даних або векторному індексі) і відкриємо його як інструменти, якими може користуватися аґент.

### Архітектура
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Сценарій 1 — Користувач, який бронює поїздку на річницю вперше

Сара відвідує вперше. Агент повинен зберігати її вподобання за допомогою інструментів і використовувати їх для рекомендації готелів.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Сценарій 2 — Сара повертається через кілька тижнів

Сара починає **зовсім нову розмову** (імітація нової сесії). Робоча пам’ять порожня, але довготривале сховище вподобань досі має її інформацію. Агент повинен отримати її та використати для персоналізації рекомендацій.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Резюме

У цьому уроці ви дізналися про три типи пам’яті агента та як їх реалізувати за допомогою Microsoft Agent Framework:

| Тип пам’яті | Механізм MAF | Тривалість |
|---|---|---|
| **Робоча** | `agent.create_session()` | Одна розмова |
| **Короткострокова** | Накопичений контекст у межах потоку | Одне завдання / сесія |
| **Довгострокова** | Зовнішнє сховище, доступне через функції `@tool` | Між сесіями |

### Ключові висновки
1. **`agent.create_session()`** забезпечує робочу пам’ять — агент бачить повну історію розмови в межах сесії.
2. **Нові сесії втрачають контекст** — без довгострокової пам’яті агент не може згадати попередні розмови.
3. **Функції `@tool`** долають розрив — вони дозволяють агенту зберігати та отримувати інформацію з постійного сховища.
4. **Персоналізація покращується з часом** — чим більше збережених вподобань, тим кращі рекомендації агента.

### Реальні застосування
- **Обслуговування клієнтів**: Запам’ятовувати історію та вподобання клієнта
- **Персональні помічники**: Підтримувати контекст протягом днів або тижнів
- **Охорона здоров’я**: Відстежувати інформацію про пацієнта та вподобання
- **Електронна комерція**: Персоналізований шопінг на основі історії

### Наступні кроки
- Замінити словник у пам’яті на базу даних або векторне сховище (наприклад, Azure AI Search)
- Додати термін дії для інформації з часовими обмеженнями
- Побудувати системи з кількома агентами із спільною пам’яттю
- Дослідити нотатник Cognee для пам’яті на основі графу знань


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
